# U9b · Modelos fundacionales — llamar por API con OpenRouter

> **Para quien pone el criterio clínico, no el código.** En el cuaderno anterior (`U09a`) **descargamos** modelos de Hugging Face y los ejecutamos en nuestro propio entorno. Aquí abrimos la **segunda vía**: los modelos más grandes y capaces —Claude, GPT, Gemini, Qwen— no se descargan, se **llaman por API**. Verás que, gracias a **OpenRouter**, llamar a 400+ modelos distintos se reduce a **una clave y una llamada**.

> ⚠️ Todos los datos de partida son **sintéticos** (generados por código, semilla fija). **No representan pacientes reales.** Esto es especialmente importante en este cuaderno: a diferencia de Hugging Face (donde el modelo venía a los datos), aquí **el dato sale de tu entorno** hacia los servidores de un tercero. Por eso, en todo este cuaderno, **solo enviaremos texto sintético**, nunca una nota real.

**Qué haremos, en clave clínica:**
1. Entender qué es **OpenRouter** y por qué, si conoces la API de OpenAI, ya sabes usarlo.
2. Comprobar si tenemos una **clave de API** disponible en el entorno — y, si no la hay, aprender **cómo conseguirla** sin que el cuaderno intente llamar a nada.
3. Si hay clave: pedirle a un modelo (por ejemplo, una variante **gratuita** de Qwen) que **resuma y clasifique** una nota clínica de `notas_clinicas.csv`.
4. Cerrar con las reglas de **privacidad** que gobiernan el uso de estas APIs en un contexto sanitario.

[Abrir en Colab](https://colab.research.google.com/drive/1MU72hahjS96Nf1I3RfXOsqJkHrSJq_OI)


## 0. Preparamos los datos (esta celda se ejecuta sola)

La primera celda de cada cuaderno del curso **genera los ficheros sintéticos** en la carpeta de trabajo, entre ellos `notas_clinicas.csv`, que usaremos como ejemplo de tarea para la API. Ejecútala una vez (▶ o *Mayús+Enter*) y sigue hacia abajo.


In [ ]:
# === Generación de los datos sintéticos del curso (se ejecuta sola) ===
# TODOS LOS DATOS SON SINTÉTICOS. No representan pacientes reales. Semilla fija -> reproducible.
import os, numpy as np, pandas as pd

def generar_datos_clinicos(carpeta="."):
    """Crea los CSV del curso si no están ya en la carpeta. Devuelve la ruta."""
    if os.path.exists(os.path.join(carpeta, "pacientes.csv")):
        return carpeta
    RNG = np.random.default_rng(2026)   # misma semilla en todo el curso

    # --- Centros ---
    AREAS = ["Valencia","Alicante","Castellón","Madrid","Barcelona","Sevilla","Bilbao","Zaragoza"]
    nc = 24
    tipo = RNG.choice(["hospital","centro de salud"], nc, p=[0.35,0.65])
    camas = np.where(tipo=="hospital", RNG.integers(120,900,nc), RNG.integers(0,25,nc))
    nserv = np.where(tipo=="hospital", RNG.integers(12,40,nc), RNG.integers(3,10,nc))
    centros = pd.DataFrame({"centro_id":[f"C{i:03d}" for i in range(1,nc+1)],"tipo":tipo,
        "area":RNG.choice(AREAS,nc),"camas":camas,"n_servicios":nserv,
        "urgencias_dia_media":np.where(tipo=="hospital",RNG.integers(80,320,nc),RNG.integers(5,40,nc)),
        "ratio_mayores65":RNG.uniform(0.12,0.34,nc).round(3)})
    centros.to_csv(os.path.join(carpeta,"centros.csv"), index=False)

    # --- Pacientes (tabla central) ---
    N = 20000
    sexo = RNG.choice(["M","F"], N, p=[0.49,0.51])
    edad = np.clip(np.where(RNG.random(N)<0.6, RNG.normal(58,15,N), RNG.normal(40,12,N)),18,95).round().astype(int)
    p_act = np.clip(0.28-0.0016*(edad-18),0.06,0.28); p_ex = np.clip(0.10+0.004*(edad-18),0.10,0.40); p_nu = 1-p_act-p_ex
    tabaquismo = np.array([RNG.choice(["nunca","ex","activo"],p=[a,b,c]) for a,b,c in zip(p_nu,p_ex,p_act)])
    actividad = RNG.choice(["baja","media","alta"], N, p=[0.4,0.4,0.2])
    antec = RNG.binomial(1,0.22,N)
    actn = pd.Series(actividad).map({"baja":1.5,"media":0.0,"alta":-1.3}).values
    imc = np.clip(RNG.normal(26.5,4.2,N)+0.03*(edad-50)+actn+RNG.normal(0,1.0,N),15,48).round(1)
    fa = (tabaquismo=="activo").astype(float); fe = (tabaquismo=="ex").astype(float)
    ta_s = np.clip(RNG.normal(120,12,N)+0.45*(edad-45)+0.7*(imc-25)+6*fa+RNG.normal(0,6,N),85,220).round().astype(int)
    ta_d = np.clip(0.55*ta_s+RNG.normal(20,6,N),50,130).round().astype(int)
    glu = np.clip(RNG.normal(100,12,N)+1.0*(imc-25)+0.3*(edad-45)+9*antec+RNG.normal(0,7,N),60,320).round(1)
    diabetes = (glu>126).astype(int)
    col = np.clip(RNG.normal(195,30,N)+0.4*(edad-45)+1.1*(imc-25)+RNG.normal(0,15,N),110,380).round(1)
    hdl = np.clip(RNG.normal(55,12,N)-0.25*(imc-25)+6*(sexo=="F")-5*fa-3*actn+RNG.normal(0,6,N),20,110).round(1)
    z = (-3.1+0.062*(edad-50)+0.019*(ta_s-120)+0.008*(col-190)-0.028*(hdl-55)+0.055*(imc-25)
         +0.85*fa+0.30*fe+0.60*diabetes+0.45*antec+0.55*(sexo=="M")
         +0.012*np.maximum(edad-65,0)*fa+RNG.normal(0,0.35,N))
    p = 1/(1+np.exp(-z))
    riesgo = (100*p).round(1); evento = RNG.binomial(1,p)
    pacientes = pd.DataFrame({"paciente_id":[f"P{i:05d}" for i in range(1,N+1)],"edad":edad,"sexo":sexo,
        "imc":imc,"ta_sistolica":ta_s,"ta_diastolica":ta_d,"glucemia":glu,"colesterol_total":col,"hdl":hdl,
        "tabaquismo":tabaquismo,"actividad_fisica":actividad,"antecedentes_familiares":antec,
        "diabetes":diabetes,"riesgo_cv_10a":riesgo,"evento_cv":evento})
    pacientes.to_csv(os.path.join(carpeta,"pacientes.csv"), index=False)

    # --- Versión "sucia" para EDA ---
    d = pacientes.copy()
    d["sexo"] = d["sexo"].map(lambda s: RNG.choice({"M":["M","m","Masculino","H","Hombre"],"F":["F","f","Femenino","Mujer"]}[s]))
    idx = RNG.choice(d.index,int(N*0.06),replace=False)
    d["glucemia"] = d["glucemia"].astype(object)   # permite mezclar texto (mmol/L con coma) y números
    d.loc[idx,"glucemia"] = [str(round(v/18.0,1)).replace(".",",") for v in d.loc[idx,"glucemia"]]
    prob_na = np.where(pacientes["edad"]<40,0.12,0.04); mask = RNG.random(N)<prob_na
    d.loc[mask,"hdl"] = np.nan
    d.loc[RNG.choice(d.index,int(N*0.05),replace=False),"colesterol_total"] = np.nan
    d.loc[RNG.choice(d.index,25,replace=False),"edad"] = RNG.integers(150,260,25)
    d.loc[RNG.choice(d.index,20,replace=False),"ta_sistolica"] = -RNG.integers(1,90,20)
    d.loc[RNG.choice(d.index,15,replace=False),"imc"] = RNG.uniform(80,200,15).round(1)
    d["colesterol_total"] = d["colesterol_total"].astype(object); d["ta_diastolica"] = d["ta_diastolica"].astype(object)
    d.loc[RNG.choice(d.index,30,replace=False),"colesterol_total"] = "desconocido"
    d.loc[RNG.choice(d.index,20,replace=False),"ta_diastolica"] = "N/D"
    d["tabaquismo"] = d["tabaquismo"].map(lambda s: RNG.choice({"nunca":["nunca","No fumador","NUNCA"],
        "ex":["ex","exfumador","Ex-fumador"],"activo":["activo","Fumador","SI"]}[s]))
    d = pd.concat([d, d.sample(400,random_state=3)], ignore_index=True)
    idx = RNG.choice(d.index,200,replace=False); d.loc[idx,"paciente_id"] = d.loc[idx,"paciente_id"].map(lambda s:" "+s+" ")
    d = d.sample(frac=1,random_state=11).reset_index(drop=True)
    d.to_csv(os.path.join(carpeta,"pacientes_sucio.csv"), index=False)

    # --- Urgencias (serie temporal) ---
    ND = 730; fechas = pd.date_range("2024-01-01", periods=ND, freq="D")
    doy = fechas.dayofyear.values; dow = fechas.dayofweek.values
    fest_set = {(1,1),(1,6),(5,1),(8,15),(10,12),(11,1),(12,6),(12,8),(12,25)}
    festivo = np.array([1 if (f.month,f.day) in fest_set else 0 for f in fechas])
    gripe = np.array([1 if f.month in (12,1,2) else 0 for f in fechas])
    temp = (15+10*np.sin(2*np.pi*(doy-110)/365)+RNG.normal(0,2.5,ND)).round(1)
    est = 1+0.14*np.sin(2*np.pi*(doy-20)/365)
    sem = np.where(dow==0,1.18,np.where(dow>=5,1.10,1.0))
    mu = 120.0*est*sem*(1+0.22*gripe)*(1+0.15*festivo)
    ingresos = RNG.poisson(np.maximum(mu,1)).astype(int)
    pd.DataFrame({"fecha":fechas.strftime("%Y-%m-%d"),"ingresos":ingresos,"festivo":festivo,
        "temporada_gripe":gripe,"temperatura":temp}).to_csv(os.path.join(carpeta,"urgencias_diarias.csv"), index=False)

    # --- Notas clínicas (texto) ---
    plant = {"cardiología":["dolor torácico opresivo de {t} de evolución, irradiado a brazo izquierdo",
        "disnea de esfuerzo progresiva y palpitaciones, antecedente de hipertensión",
        "control de anticoagulación, fibrilación auricular conocida, sin dolor actual",
        "edemas en miembros inferiores y ortopnea, sospecha de insuficiencia cardiaca"],
      "respiratorio":["tos productiva de {t}, fiebre y disnea, auscultación con crepitantes",
        "sibilancias y opresión torácica, antecedente de asma, mala respuesta a inhalador",
        "disnea súbita y dolor pleurítico, saturación disminuida",
        "tos seca persistente y febrícula, contacto con caso respiratorio"],
      "digestivo":["dolor abdominal en fosa iliaca derecha de {t}, náuseas y febrícula",
        "epigastralgia y pirosis, relación con las comidas, sin signos de alarma",
        "diarrea de {t} sin productos patológicos, buen estado general",
        "ictericia y coluria, dolor en hipocondrio derecho"],
      "neurología":["cefalea intensa de inicio súbito, la peor de su vida, con fotofobia",
        "focalidad neurológica de {t}, debilidad en hemicuerpo y disartria",
        "mareo con giro de objetos y vómitos, sin cortejo vegetativo",
        "crisis comicial presenciada, recuperación progresiva del nivel de conciencia"],
      "traumatología":["caída casual con dolor e impotencia funcional en muñeca, deformidad",
        "lumbalgia mecánica de {t} tras esfuerzo, sin déficit neurológico",
        "esguince de tobillo con edema y dificultad para la deambulación",
        "gonalgia y derrame articular tras traumatismo deportivo"]}
    prio = {"cardiología":[0.35,0.45,0.20],"respiratorio":[0.30,0.45,0.25],"digestivo":[0.20,0.50,0.30],
        "neurología":[0.45,0.35,0.20],"traumatología":[0.12,0.48,0.40]}
    tiempos = ["horas","un día","dos días","una semana","varios días"]; esp_list = list(plant.keys()); rows=[]
    for _ in range(3000):
        e = RNG.choice(esp_list); txt = RNG.choice(plant[e]).replace("{t}",RNG.choice(tiempos))
        rows.append((txt,e,RNG.choice(["alta","media","baja"],p=prio[e]),RNG.choice(centros["centro_id"].values)))
    pd.DataFrame(rows,columns=["texto","especialidad","prioridad","centro_id"]).to_csv(os.path.join(carpeta,"notas_clinicas.csv"),index=False)

    # --- Wearable (señal) ---
    rows=[]
    for s in range(1,201):
        fcb=RNG.normal(66,6); pb=RNG.normal(7500,2500); sb=RNG.normal(7.0,0.8); anom=RNG.random()<0.05
        for dia in range(1,31):
            fc=fcb+RNG.normal(0,3); pasos=max(0,pb+RNG.normal(0,1500)); sue=float(np.clip(sb+RNG.normal(0,0.6),3,11))
            if anom and dia in (14,15,16): fc+=RNG.uniform(18,30); sue-=RNG.uniform(1.5,3)
            rows.append((f"S{s:03d}",dia,round(fc,1),int(pasos),round(sue,1)))
    pd.DataFrame(rows,columns=["sujeto_id","dia","fc_reposo","pasos","horas_sueno"]).to_csv(os.path.join(carpeta,"wearable.csv"),index=False)
    return carpeta

generar_datos_clinicos(".")
print("Datos sintéticos listos en la carpeta actual. (Recuerda: son datos SINTÉTICOS, no reales.)")

## 1. OpenRouter: un único punto de acceso a 400+ modelos

Los modelos de lenguaje más capaces —los que hay detrás de los asistentes conversacionales de referencia: **Claude** (Anthropic), **GPT** (OpenAI), **Gemini** (Google)— son, en su mayoría, **propietarios**: no se descargan, se **usan a distancia** a través de una API. Le mandas tu petición a los servidores del proveedor, el modelo la procesa allí y te devuelve la respuesta. El problema clásico es la **fragmentación**: cada proveedor tiene su propia cuenta, su propia clave y su propia forma de llamar. Probar cinco modelos de cinco proveedores distintos es un engorro.

**OpenRouter** resuelve esto siendo un **agregador**: un único servicio que da acceso a **más de 400 modelos** de múltiples proveedores (Claude, GPT, Gemini, Qwen y muchos más) con **una sola clave** y **una sola factura**. Su ventaja práctica más importante: es **compatible con la API de OpenAI**. Esto significa que si ya sabes (o tu asistente de IA sabe) llamar a la API de OpenAI con la librería `openai`, **ya sabes usar OpenRouter**: el código es idéntico, solo cambias **dos cosas** — la dirección del servicio (`base_url`) y la clave (`api_key`).


{% hint style="info" %}
**Concepto · OpenRouter (un endpoint para muchos modelos)**

Servicio que ofrece un **único punto de acceso** a 400+ modelos de múltiples proveedores, con **una sola clave**. Es **compatible con la API de OpenAI**: mismo código, cambiando solo la `base_url` (a `https://openrouter.ai/api/v1`) y la clave. Tiene modelos **gratuitos** (identificados con el sufijo `:free`) y una capa de pago con un pequeño recargo (~5,5 %) sobre el precio del proveedor.
{% endhint %}


El patrón de código, cuando ya se tiene una clave, es siempre este:

```python
from openai import OpenAI

cliente = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=API_KEY,   # nunca la escribas a mano en el notebook
)

respuesta = cliente.chat.completions.create(
    model="qwen/...:free",   # el identificador exacto del catálogo de OpenRouter
    messages=[
        {"role": "system", "content": "Eres un asistente de análisis de datos clínicos."},
        {"role": "user",   "content": "Resume en 3 puntos ..."},
    ],
)
print(respuesta.choices[0].message.content)
```

Fíjate en que es literalmente el mismo SDK `openai` que se usa para hablar con OpenAI directamente. Vamos a construir esto paso a paso, empezando por comprobar que tenemos la librería instalada.


### 1.1 Instalamos la librería `openai` (si hiciera falta)

En Google Colab, la librería `openai` puede no venir preinstalada. Esta celda comprueba si está disponible y, si no, intenta instalarla. Está envuelta en `try/except` para que, aunque falte conexión, el cuaderno **no se detenga**.


In [ ]:
try:
    import openai
    print(f"openai ya está disponible (versión {openai.__version__}).")
except Exception as e:
    print("openai no estaba disponible; probamos a instalarlo con pip...")
    try:
        import subprocess
        subprocess.run(["pip", "install", "-q", "openai"], check=True)
        import openai
        print(f"Instalado. Versión {openai.__version__}.")
    except Exception as e2:
        print("No se ha podido instalar openai en este entorno.")
        print("Motivo:", repr(e2))
        print("En Colab, ejecuta en una celda: !pip install openai")

Con la librería lista, ya podemos pasar a la parte que de verdad importa: **la clave**.


## 2. La clave de API: cómo se consigue y cómo se guarda (nunca en el código)

Antes de poder llamar a cualquier modelo por OpenRouter, hace falta una **clave de API** (*API key*). Es un fragmento de texto secreto, único para tu cuenta, que identifica quién hace la llamada (y, si usas modelos de pago, quién la paga). Una regla que no tiene excepción: **una clave nunca se escribe a mano dentro de un notebook**. Si el cuaderno se comparte, se sube a un repositorio o simplemente lo ve otra persona por encima del hombro, la clave quedaría expuesta.

La forma correcta de manejarla es guardarla como **variable de entorno** o como **secreto** de Colab, y leerla desde el código con `os.environ.get(...)` — así el propio texto de la clave **nunca aparece** en el notebook.

**Cómo conseguir una clave de OpenRouter (si todavía no tienes una):**
1. Entra en [openrouter.ai](https://openrouter.ai) y crea una cuenta (puedes usar Google o GitHub).
2. Ve a la sección **Keys** de tu perfil y pulsa **Create Key**.
3. Copia la clave (empieza por `sk-or-...`) — solo se muestra una vez.
4. En Google Colab: abre el panel lateral **🔑 Secretos**, crea uno llamado `OPENROUTER_API_KEY`, pega la clave y activa el acceso del notebook a ese secreto. En tu propio ordenador, puedes definir la variable de entorno `OPENROUTER_API_KEY` antes de lanzar Jupyter.
5. Hay modelos con el sufijo `:free` que no consumen saldo — perfectos para practicar sin coste.


In [ ]:
import os

# Leemos la clave del entorno (Colab: Secretos; local: variable de entorno). NUNCA la escribimos aquí.
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")

if OPENROUTER_API_KEY:
    print("Clave OPENROUTER_API_KEY encontrada en el entorno. Longitud:", len(OPENROUTER_API_KEY), "caracteres.")
    print("(Por seguridad, no imprimimos la clave en sí.)")
else:
    print("No se ha encontrado ninguna clave OPENROUTER_API_KEY en este entorno.")
    print()
    print("Para conseguir una y poder ejecutar el resto del cuaderno con una llamada real:")
    print("  1. Crea una cuenta en https://openrouter.ai")
    print("  2. Ve a tu perfil -> 'Keys' -> 'Create Key' y copia la clave (empieza por 'sk-or-...')")
    print("  3. En Colab: panel lateral 'Secretos' -> nuevo secreto 'OPENROUTER_API_KEY' -> pega la clave")
    print("     y activa el acceso del notebook a ese secreto. Vuelve a ejecutar esta celda.")
    print("  4. Elige un modelo con sufijo ':free' para no gastar saldo mientras aprendes.")
    print()
    print("Sin clave, las siguientes celdas NO harán ninguna llamada real: solo explicarán qué habrían hecho.")

**Lo que acabamos de hacer:** comprobar, sin arriesgar nada, si el entorno ya tiene una clave lista para usar. Si la tienes, genial: las siguientes celdas harán una llamada real. Si no la tienes, el mensaje te dice exactamente los pasos para conseguirla — y, mientras no la tengas, **el resto del cuaderno no intentará llamar a ningún servidor**.


## 3. Usar la API para apoyar la lectura de una nota clínica

Vamos a construir la tarea de ejemplo: le pediremos al modelo que **resuma y clasifique** una nota de nuestro `notas_clinicas.csv` (recuerda: **sintética**). Primero elegimos la nota; la llamada al modelo va en el bloque siguiente, y solo se ejecutará si hay clave disponible.


In [ ]:
import pandas as pd

notas = pd.read_csv("notas_clinicas.csv")

# Elegimos una nota de ejemplo (sintética) para pedirle al modelo que la resuma y clasifique
nota_ejemplo = notas.iloc[0]
print("Nota clínica de ejemplo (SINTÉTICA):")
print(" Texto:       ", nota_ejemplo["texto"])
print(" Especialidad:", nota_ejemplo["especialidad"], "  (esta es la respuesta 'real' que ya conocemos)")
print(" Prioridad:   ", nota_ejemplo["prioridad"])
print()
print("Le pediremos al modelo que, leyendo solo el texto, proponga la especialidad y la prioridad,")
print("para comparar su propuesta con la que ya sabemos por el propio dataset.")

{% hint style="danger" %}
**⚠️ Privacidad: por esta vía, solo datos sintéticos**

La nota que vamos a enviar es **sintética**: no corresponde a ningún paciente real. Esto no es un detalle menor. Vamos a llamar a un servidor de un tercero (el proveedor del modelo, a través de OpenRouter), y **el texto que le pasemos viaja hasta allí**. Enviar una nota clínica real, con o sin nombre, sería un problema serio de privacidad sin las garantías legales y contractuales adecuadas. Regla práctica: a una API pública, **solo datos sintéticos, anonimizados o agregados**. Si necesitas procesar una nota real con un modelo de lenguaje, la vía es la del cuaderno anterior — un modelo **local** de Hugging Face — o un despliegue con las debidas garantías dentro de tu organización.
{% endhint %}


In [ ]:
import os

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")

if not OPENROUTER_API_KEY:
    print("No hay clave OPENROUTER_API_KEY en el entorno: NO se realiza ninguna llamada real.")
    print("(Repasa la sección 2 para conseguir una clave gratuita y repetir esta celda.)")
    print()
    print("Si hubiera clave, este es el resultado que cabría esperar de un modelo capaz:")
    print(' Especialidad propuesta: "', nota_ejemplo["especialidad"], '" (o similar)')
    print(' Prioridad propuesta:    "', nota_ejemplo["prioridad"], '" (o similar)')
    print(" + un resumen en una frase de la nota, en lenguaje llano.")
else:
    try:
        from openai import OpenAI

        cliente = OpenAI(
            base_url="https://openrouter.ai/api/v1",
            api_key=OPENROUTER_API_KEY,
        )

        pregunta = (
            "Esta es una nota clínica SINTÉTICA de un curso de formación (no es un paciente real):\n"
            f'"{nota_ejemplo["texto"]}"\n\n'
            "1. Resúmela en una frase, en lenguaje llano.\n"
            "2. Propón la especialidad más probable (una palabra).\n"
            "3. Propón la prioridad más probable: alta, media o baja."
        )

        respuesta = cliente.chat.completions.create(
            model="qwen/qwen-2.5-72b-instruct:free",   # variante gratuita; cambia el modelo si no está disponible
            messages=[
                {"role": "system", "content": "Eres un asistente de apoyo al análisis de notas clínicas sintéticas."},
                {"role": "user", "content": pregunta},
            ],
        )
        print("Respuesta del modelo:")
        print(respuesta.choices[0].message.content)

    except Exception as e:
        print("Había clave, pero la llamada a OpenRouter no se ha podido completar en este entorno.")
        print("Motivo:", repr(e))
        print("Causas típicas: sin conexión a internet, el modelo ':free' elegido está temporalmente saturado,")
        print("o el identificador de modelo ha cambiado en el catálogo de OpenRouter (revísalo en openrouter.ai/models).")

**Lo que hemos hecho:** si había clave, hemos mandado una nota **sintética** a un modelo de última generación (una variante gratuita de Qwen, en este ejemplo) y le hemos pedido que la resuma y proponga especialidad y prioridad — exactamente el tipo de tarea de **apoyo al análisis** para la que tiene sentido usar estas APIs: no como sustituto del criterio clínico, sino como un copiloto que redacta, resume o propone una primera lectura que el profesional revisa. Si no había clave, el cuaderno te ha explicado igualmente qué cabría esperar, sin intentar ninguna conexión.

**Fíjate en el patrón de código:** es exactamente el SDK `openai` de siempre. Cambiando el nombre del `model=` por el identificador de otro modelo del catálogo de OpenRouter (por ejemplo, uno de Claude o de GPT), **el resto del código no cambia una línea**. Esa es la gran ventaja de que OpenRouter sea compatible con la API de OpenAI.


## 4. Qwen, y cuándo usar cada vía

En el ejemplo anterior elegimos deliberadamente una variante **gratuita de Qwen**, la familia de modelos de **Alibaba**. Merece la pena saber por qué: buena parte de Qwen se publica con **pesos abiertos** (licencia Apache 2.0), desde variantes pequeñas que corren en hardware modesto hasta modelos grandes de gran capacidad. Eso significa que Qwen aparece en **las dos vías** del curso: se puede **descargar** de Hugging Face y ejecutar en local (como en `U09a`), o se puede **llamar por API** vía OpenRouter sin instalar nada (como aquí) — a menudo gratis o casi. Es el ejemplo perfecto de que, hoy, la potencia de primer nivel ya no depende del presupuesto, sino del **criterio** para elegir y usar bien el modelo.

Con los dos cuadernos ya recorridos, el mapa de decisión completo queda así:


| Situación / necesidad | Vía recomendada | Por qué |
| --------------------- | --------------- | ------- |
| Dato **sensible** de paciente, no puede salir | **Hugging Face local** (`U09a`) | El modelo va a los datos; nada sale de tu entorno |
| Existe un **modelo especializado** (imagen médica, texto clínico) | **Hugging Face** (`pipeline()`) | Modelo ya afinado para tu tarea, en tres líneas |
| Necesitas **la máxima capacidad** de lenguaje ya | **API vía OpenRouter** (`U09b`) | Acceso inmediato a Claude/GPT/Gemini sin hardware |
| **Prototipar** rápido sobre datos sintéticos/agregados | **API vía OpenRouter** (`U09b`) | Una clave, una llamada, 400+ modelos |
| Quieres **potencia abierta y barata** | **Qwen** (abierto: local o API) | Pesos abiertos Apache 2.0, de <1B a modelos grandes |


{% hint style="warning" %}
**⚠️ Reproducibilidad: el proveedor puede cambiar el modelo sin avisar**

A diferencia de un modelo descargado de Hugging Face (donde tú fijas la versión exacta), un modelo servido por API puede **actualizarse** por el proveedor en cualquier momento. Una misma pregunta puede dar respuestas distintas con el tiempo. Para cualquier uso serio, **anota el identificador exacto del modelo** que usaste (incluida la fecha) — es parte de la trazabilidad que la investigación y la clínica exigen.
{% endhint %}


{% hint style="success" %}
**💡 Idea clave**

El resumen del cuaderno cabe en una frase: **con una sola clave, y sin gestionar cinco cuentas, tienes acceso a los modelos más capaces del momento** — pero esa comodidad tiene un precio en privacidad que solo tú puedes gestionar bien. La regla no es "no usar nunca estas APIs en salud": es **usarlas solo con datos que puedan salir de tu entorno** (sintéticos, anonimizados, agregados) y reservar los datos reales sensibles para la Vía A (modelos locales) o para despliegues con garantías dentro de tu organización.
{% endhint %}


## 5. Qué hemos aprendido

- **OpenRouter da acceso a 400+ modelos con una sola clave**, siendo compatible con la API de OpenAI: el mismo código de siempre, cambiando solo `base_url` y `api_key`.
- **La clave nunca se escribe en el notebook.** Se lee con `os.environ.get("OPENROUTER_API_KEY")`, guardada como secreto de Colab o variable de entorno. Sin clave, el cuaderno explica cómo conseguirla y no intenta ninguna llamada.
- **Probamos la tarea de "copiloto de análisis"**: pedirle a un modelo (una variante gratuita de Qwen) que resuma y clasifique una nota clínica **sintética** de `notas_clinicas.csv`.
- **Qwen es el puente entre las dos vías del curso**: pesos abiertos, descargable de Hugging Face o accesible por API — potencia de primer nivel a coste casi nulo.
- **La regla de privacidad no admite excepciones en los ejercicios**: a una API pública, solo datos sintéticos, anonimizados o agregados. Nunca una nota o un dato identificable de un paciente real.

**Con esto se cierra el círculo de la U9**: ya sabes las dos formas de "usar" un modelo fundacional ya entrenado — descargarlo de Hugging Face (`U09a`) o llamarlo por API con OpenRouter (`U09b`) — y, sobre todo, sabes que la parte fácil es ejecutar; la que de verdad importa es decidir **cuál conviene, cuándo, y con qué datos**. La última pieza del curso, en la **U10**, es aprender a dirigir todo esto de forma fluida: convertir a la IA en tu copiloto de ciencia de datos de principio a fin.


## 6. Y esto, ¿cómo se lo pido a un asistente de IA?

Recuerda el principio del curso: **tú pones el criterio clínico, la IA escribe el código.** Un buen encargo para reproducir (o ampliar) este cuaderno sería:

> *"En español y por celdas, usando la librería `openai` apuntando a OpenRouter (`base_url='https://openrouter.ai/api/v1'`): (1) explícame en dos frases por qué OpenRouter, al ser compatible con la API de OpenAI, me permite usar 400+ modelos cambiando solo dos líneas de código. (2) Escribe el código para leer la clave desde `os.environ.get('OPENROUTER_API_KEY')`; si no existe, que me explique cómo conseguirla en openrouter.ai y NO intente ninguna llamada. (3) Si hay clave, pide a una variante gratuita de un modelo (por ejemplo, Qwen) que resuma en una frase y proponga la especialidad y prioridad de una nota clínica SINTÉTICA que yo te dé como texto. (4) Recuérdame, antes de ejecutar nada, la regla de privacidad: por esta vía solo pueden viajar datos sintéticos o anonimizados, nunca datos reales de un paciente."*

Fíjate en el punto (4): no le pides solo que ejecute la llamada, le pides que **te recuerde el límite antes de cruzarlo**. Ese es, otra vez, el criterio que no se automatiza — y el que separa un uso responsable de la IA de uno ingenuo.
